In [32]:
%pip install "langchain>=1,<2" "langchain-text-splitters>=1,<2" "langchain-huggingface>=1,<2" "sentence-transformers>=5,<6" "transformers==4.56.2" "torch>=2.4",


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
%pip install pymupdf
%pip install -U langchain-ollama

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip



   ---------------------------------------- 0/2 [ollama]
   -------------------- ------------------- 1/2 [langchain-ollama]
   ---------------------------------------- 2/2 [langchain-ollama]

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import re
import sys
from pathlib import Path
import pymupdf  # PyMuPDF
import torch
import fitz
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from transformers import AutoModelForCausalLM, AutoTokenizer


def cargar_y_fragmentar(rutas_archivos: list[str]) -> list[Document]:
    documentos_base = []

    for ruta in rutas_archivos:
        file_path = Path(ruta)
        nombre_fuente = file_path.stem  # Ej. "rgepa" a partir de "rgepa.pdf"

        if file_path.suffix.lower() == ".pdf":
            with fitz.open(str(file_path)) as pdf:
                texto = "\n".join(pagina.get_text("text") for pagina in pdf)
        elif file_path.suffix.lower() in (".txt", ".md"):
            texto = file_path.read_text(encoding="utf-8")
        else:
            print(f"Formato no soportado: {ruta}")
            continue

        if texto.strip():
            documentos_base.append(
                Document(
                    page_content=texto,
                    metadata={"reglamento": nombre_fuente, "fuente": str(file_path)}
                )
            )

    # Cortamos con el splitter oficial de LangChain por Artículo
    splitter = RecursiveCharacterTextSplitter(
        separators=[
            r"\n\s*(?:Artículo|ARTÍCULO)\s+\d+",
            r"\n\s*(?:CAPÍTULO|Capítulo)\s+[IVXLCDM\d]+",
            "\n\n",
            "\n",
            " "
        ],
        keep_separator="start",
        is_separator_regex=True,
        chunk_size=1200,
        chunk_overlap=100
    )
    fragmentos = splitter.split_documents(documentos_base)

    # Extraemos el número de artículo para que metadata["articulo"] siempre exista
    for frag in fragmentos:
        match = re.search(r"(?:Artículo|ARTÍCULO)\s+(\d+(?:o|°)?(?:\s+(?:Bis|Ter))?)", frag.page_content)
        frag.metadata["articulo"] = f"Art. {match.group(1)}" if match else "Encabezado/General"

    return fragmentos


dispositivo = "cuda" if torch.cuda.is_available() else "cpu"  # Elige GPU CUDA si está disponible.
print("Python:", sys.version.split()[0], "| Dispositivo:", dispositivo)  #[cite: 1]

# Fuentes:
# https://myengineeringpath.dev/programming/python/rag-pipeline-tutorial/
# https://reference.langchain.com/python/langchain-text-splitters/character/RecursiveCharacterTextSplitter

Python: 3.14.7 | Dispositivo: cpu


C:\Users\EQUIPO\desktop\chatbot_udg_proyecto\env\Lib\site-packages\fitz\__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(


In [4]:

mis_archivos = ["archivos/revalidaciones_equivalencias_acreditacion.pdf"] 
fragmentos = cargar_y_fragmentar(mis_archivos)

print(f"Total de fragmentos (artículos) generados: {len(fragmentos)}\n")
print("-" * 60)


for numero, frag in enumerate(fragmentos[:5], start=1):
   
    reglamento = frag.metadata.get("reglamento", "N/A")
    longitud = len(frag.page_content)
    

    vista_previa = frag.page_content[:140].replace("\n", " ")
    
    print(f"{numero:02d} | [{reglamento}] | {longitud:4d} chars | {vista_previa}...")

Total de fragmentos (artículos) generados: 18

------------------------------------------------------------
01 | [revalidaciones_equivalencias_acreditacion] | 1062 chars | REGLAMENTO DE REVALIDACIONES, ESTABLECIMIENTO DE EQUIVALENCIAS Y ACREDITACIÓN DE ESTUDIOS 2017 1 REGLAMENTO DE REVALIDACIONES, ESTABLECIMIEN...
02 | [revalidaciones_equivalencias_acreditacion] |  896 chars | Artículo 5. Los estudios realizados fuera del sistema educativo nacional podrán adquirir validez oficial, mediante su revalidación, siempre ...
03 | [revalidaciones_equivalencias_acreditacion] |  928 chars | Artículo 8. En el caso de personas que soliciten revalidación parcial para concluir estudios de licenciatura, maestría o de doctorado y haya...
04 | [revalidaciones_equivalencias_acreditacion] | 1182 chars | Artículo 11. Las solicitudes de revalidación se acompañarán de los siguientes documentos: I. En el caso de revalidación parcial para conclui...
05 | [revalidaciones_equivalencias_acreditacion] |  478 char

In [5]:

print("METADATOS:", fragmentos[1].metadata)
print("\nTEXTO COMPLETO DEL FRAGMENTO:\n")
print(fragmentos[10].page_content)

METADATOS: {'reglamento': 'revalidaciones_equivalencias_acreditacion', 'fuente': 'archivos\\revalidaciones_equivalencias_acreditacion.pdf', 'articulo': 'Art. 5'}

TEXTO COMPLETO DEL FRAGMENTO:

Artículo 23. Las solicitudes de establecimiento de equivalencias de estudios se
presentarán por escrito ante la Secretaría Administrativa del Centro Universitario
respectivo o del Sistema de Educación Media Superior antes o después de que los
interesados hayan sido admitidos por la Universidad.
Artículo 24. Las solicitudes de establecimiento de equivalencias de estudios de
bachillerato, licenciatura o posgrado se acompañarán de los siguientes documentos:
I.
Certificado de estudios legalizado, si es el caso;
II.
Plan de estudios con el que se pretende establecer equivalencia; y
III.
Programas, temarios o cualquier otra documentación que exprese los
contenidos de cada unidad de enseñanza-aprendizaje, con el que se
pretende establecer equivalencia.
Artículo
25.
Recibida
la
solicitud
y la
documentac

In [6]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    model_kwargs={"device": dispositivo},
    encode_kwargs={"normalize_embeddings": True},
)
vector = embeddings.embed_query("¿Cuales documentos debo llevar para realizar mi revalidacion?")  # Convierte la consulta a un vector.
print("Dimensión:", len(vector))
print("Primeras cinco coordenadas:", [round(valor, 4) for valor in vector[:5]])

indice = InMemoryVectorStore(embedding=embeddings)  # Crea el índice en RAM.
indice.add_documents(fragmentos)  # Indexa texto, vectores y metadatos.

Dimensión: 1024
Primeras cinco coordenadas: [-0.0248, 0.0119, -0.0577, -0.0042, -0.0075]


['681bc1e1-45ea-4965-a56a-687a1d1d9d19',
 '10b25892-371b-4b1a-86ca-7a5af75bff4f',
 'c8d015f4-9dfd-4720-a84c-cdac180557dd',
 'd2760922-e416-47af-b325-7ea3f945b5fe',
 'e6ead470-120f-45f3-8d47-3de9242f6e38',
 '592d7afd-57c4-406e-94d6-7a67132fa177',
 '69ac95f8-5ffb-4f9b-b6a2-bb651f708337',
 '39f20279-a422-490f-bc92-e7f74dcbf0ed',
 'b5032e6a-c635-42ee-b98e-bef8c42ac968',
 '7ebe3825-309b-4976-a43d-f24074b2adb1',
 '2dee9ff2-6013-47ba-baa6-3ed536ee3616',
 '02d1c1b4-d36f-42a8-a146-a1fef9c9f811',
 '565e46d8-23a2-40e3-978a-25caf7f2ab3c',
 'a25ad57d-335a-4a5a-9830-52d54450a7e1',
 '69594617-e472-4f58-962f-d4a104df2713',
 'fb710730-6293-4ff3-88cd-3668fae727b8',
 '106311b3-8a27-4b90-9f09-92e2ba11da06',
 'e4515dff-6aec-4967-bc21-3f3f98a8c505']

In [7]:
pregunta = "¿Como puedo revalidar materias realizadas en el extranjero?"
for fragmento, similitud in indice.similarity_search_with_score(pregunta, k=3):  # Recupera hasta tres coincidencias.
    print(f'{similitud:.3f} | {fragmento.metadata["articulo"]} | {fragmento.page_content}')

0.619 | Art. 5 | Artículo 5. Los estudios realizados fuera del sistema educativo nacional podrán
adquirir validez oficial, mediante su revalidación, siempre y cuando sean equiparables
con estudios que vigentes en la Universidad de Guadalajara.
Artículo 6. La revalidación podrá otorgarse por niveles educativos, por grados
escolares o por asignaturas u otras unidades de enseñanza-aprendizaje, de conformidad
con el plan de estudios correspondiente.
Artículo 7. La revalidación de estudios puede ser total o parcial. La revalidación
parcial tiene el exclusivo propósito de permitir la conclusión de los estudios del nivel
medio superior y superior que se imparten en la Universidad de Guadalajara. La
revalidación total se realizará exclusivamente para efectos de ingreso a estudios de
licenciatura y posgrado en la Universidad de Guadalajara y, como consecuencia, para los
ulteriores efectos de registro oficial.
0.600 | Encabezado/General | REGLAMENTO DE REVALIDACIONES, ESTABLECIMIENTO DE
EQUIVALE

In [11]:
plantilla = ChatPromptTemplate.from_messages([  # Define la plantilla de conversación.
    ("system", "Contesta la pregunta en una sola oración en español. "
               "Usa únicamente el fragmento que contiene el dato y cita su artículo. "
               "Si ningún fragmento contiene el dato, di: No hay información suficiente."),
    ("human", "Fragmentos: [Art. 90] La credencial se recoge en recepción. "
              "[Art. 91] El curso inicia en marzo.\nPregunta: ¿Dónde recojo mi credencial?"),
    ("ai", "La credencial se recoge en recepción. [Art. 90]"),  # Ejemplo de respuesta con evidencia.
    ("human", "Fragmentos: [Art. 90] La credencial se recoge en recepción."
              "\nPregunta: ¿Cuánto cuesta la credencial?"),
    ("ai", "No hay información suficiente."),  # Ejemplo de respuesta sin evidencia.
    ("human", "Fragmentos del reglamento:\n{contexto}\n\nPregunta: {pregunta}"),
])

modelo_id = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizador = AutoTokenizer.from_pretrained(modelo_id)  # Carga el tokenizador de Qwen.
modelo = AutoModelForCausalLM.from_pretrained(  # Carga el modelo generativo.
    modelo_id, dtype=torch.float16 if dispositivo == "cuda" else torch.float32,
).to(dispositivo)
modelo.eval()  # Activa el modo de evaluación.

@torch.inference_mode()  # Desactiva el registro de gradientes.
def generar(prompt):
    roles = {"system": "system", "human": "user", "ai": "assistant"}  # Traduce los roles del chat.
    mensajes = [
        {"role": roles[mensaje.type], "content": mensaje.content}
        for mensaje in prompt.to_messages()
    ]
    entradas = tokenizador.apply_chat_template(  # Formatea y tokeniza los mensajes.
        mensajes, add_generation_prompt=True, tokenize=True,
        return_dict=True, return_tensors="pt",
    ).to(dispositivo)  # Mueve los tensores al dispositivo.
    salida = modelo.generate(**entradas, max_new_tokens=64, do_sample=False)  # Genera hasta 64 tokens nuevos.
    nuevos_tokens = salida[0, entradas["input_ids"].shape[1]:]  # Retira los tokens del prompt.
    return tokenizador.decode(nuevos_tokens, skip_special_tokens=True).strip()  # Convierte la continuación a texto.